 # Hybrid search for POC terms: Barriere - fine tuned
 

### 1. all results ordered by distance -> determine cosine distance threshold

In [1]:
from torch.nn.init import xavier_normal

query_text_sauberkeit = (
    "Sauberkeit: Erwähnung von sauber, dreckig, schmutzig, hygienisch, unhygienisch, "
    "gepflegt, ungepflegt, rein, unrein, ordentlich, unordentlich, Müll, Gestank, Geruch, "
    "saubere Umgebung, schmutzige Umgebung."
)

query_text_akustik = (
    "Akustik: Erwähnung von laut, leise, Geräuschpegel, Lärm, Stille, ruhige Atmosphäre, "
    "laute Umgebung, hallend, dumpf, klarer Klang, schlechter Klang, Musiklautstärke, "
    "Hintergrundgeräusche."
)



query_text_barrierefreiheit = (
    "Barrierefreiheit: Erwähnung von barrierefrei, Rollstuhlzugang, Aufzug vorhanden, "
    "kein Aufzug, Stufen, Treppen, behindertengerecht, leicht zugänglich, schwer zugänglich, "
    "für Menschen mit Behinderung geeignet, Zugang für Rollstuhlfahrer, barrierearme Umgebung, Rollstuhlfahrer"
)

keywords_barrierefreiheit = ['barrierefreiheit', 'behindertengerecht', 'rollstuhl']



query_text_erreichbarkeit = (
    "Erreichbarkeit: Erwähnung von gut erreichbar, schwer erreichbar, mit öffentlichen "
    "Verkehrsmitteln erreichbar, Parkplatz, Parkmöglichkeiten, Anfahrt, Lage, nahe gelegen, "
    "weit entfernt, Haltestelle in der Nähe, Verkehrsanbindung, Wegbeschreibung."
)


query_text_kaffee = (
    "Kaffee: Erwähnung von Kaffee, Espresso, Cappuccino, Latte Macchiato, Kaffeegeschmack, "
    "Kaffeebohnen, frisch gebrüht, bitter, mild, Kaffeequalität, Kaffeehaus, Barista, "
    "guter Kaffee, schlechter Kaffee, Kaffeetasse, Koffeingehalt."
)


### example 
query_text_preis = (
    "Preis: Erwähnung des Preises, der Kosten, ob etwas teuer oder günstig war oder gratis, "
    "Preis-Leistungs-Verhältnis, ob jemand zu viel bezahlt hat, ob es sich gelohnt hat, "
    "überteuert, preiswert, billig, angemessen, nicht wert, günstig."
)


In [4]:
import psycopg2
import os
import pandas as pd
from sentence_transformers import SentenceTransformer
from dotenv import load_dotenv

load_dotenv()

model = SentenceTransformer("paraphrase-multilingual-MiniLM-L12-v2")

# TODO: replace query text
vector = model.encode(query_text_barrierefreiheit)
vector_str = str(vector.tolist())


# TODO: replace keywords
like_clauses = " OR ".join([f"LOWER(aspect) = '{kw}'" for kw in keywords_barrierefreiheit])

sql = f"""
SELECT review_id, aspect, aspect_id, sentiment, confidence, snippet_4,
       embedding_4  <=> '{vector_str}'::vector AS distance
FROM aspect_sentiment_results
WHERE embedding_4 <=> '{vector_str}'::vector < 0.75
OR {like_clauses}
ORDER BY distance


"""

conn = psycopg2.connect(
    host=os.getenv("DB_HOST"),
    port=os.getenv("DB_PORT"),
    dbname=os.getenv("DB_NAME"),
    user=os.getenv("DB_USER"),
    password=os.getenv("DB_PASSWORD"),
)

with conn.cursor() as cur:
    cur.execute(sql)
    rows = cur.fetchall()
    colnames = [desc[0] for desc in cur.description]

conn.close()

df_thresh = pd.DataFrame(rows, columns=colnames)
df_thresh


,review_id,aspect,aspect_id,sentiment,confidence,snippet_4,distance
0,6252952,Eintritt,6252952_1,Positive,0.9978,Wege sind Rollstuhlgängig . Eintritt kostenfrei .,0.264304
1,6385403,Zimmer,6385403_1,Negative,0.9893,Zimmer für Rollstuhlbehinderte nicht geeignet,0.276487
2,6385403,Zimmer,6385403_3,Negative,0.9893,Zimmer für Rollstuhlbehinderte nicht geeignet,0.276487
3,5109712,Szálloda,5109712_1,Positive,0.9657,"Szálloda , ahol a mozgáskorlátozottak",0.279209
4,2589597,parking,2589597_2,Positive,0.9981,accessibilité pour handicapés et parking au top,0.293806
...,...,...,...,...,...,...,...
382023,747007,Personal,747007_2,Neutral,0.5209,Bedienung ... sehr hilfsbereites Personal Anso...,0.750000
382024,908030,Barrierefreiheit,908030_1,Positive,0.9980,Elektrogerät sehr gut . Barrierefreiheit top !...,0.799229
382025,5679778,ROLLSTUHL,5679778_3,Neutral,0.8905,. nette Bedingung . ROLLSTUHL ♿ gängig auf der,0.833815
382026,5679778,ROLLSTUHL,5679778_6,Neutral,0.8905,. nette Bedingung . ROLLSTUHL ♿ gängig auf der,0.833815


In [5]:
df_thresh.shape

(382028, 7)

In [6]:
df_thresh["review_id"].nunique()

262768

### which aspects have been classified by keyword vs embedding match in hybrid retrieval

In [19]:
df_thresh['match_type'] = df_thresh.apply(
    lambda row: 'keyword' if any(kw in row['aspect'].lower() for kw in keywords_barrierefreiheit)
                else 'embedding' if row['distance'] < 0.75
                else 'none',
    axis=1
)

# TODO: update here keywords 


In [20]:
df_keyword = df_thresh[df_thresh['match_type'] == 'keyword']
df_keyword.shape

(79, 9)

In [21]:
df_keyword["aspect"].unique()

array(['Rollstuhlplätze', 'Rollstuhlparkplätze', 'Rollstuhlparkplatz',
       'Rollstuhl', 'ein Rollstuhl', 'Rollstuhl toilette',
       'Rollstuhlfahrer', 'Barrierefreiheit', 'Rollstuhl Lift',
       'Rollstuhlzentrum', 'Rollstuhlfahrern', 'Rollstuhlgängigkeit',
       'Rollstuhl Fahrer', 'Rollstuhl WC', 'Elektrorollstuhl',
       'Rollstuhl Toiletten', 'Rollstuhltaxi', 'Elektro Rollstuhl',
       'Rollstuhlauswahl', 'Rollstuhlrampe', 'Rollstuhl - Toilette',
       'rollstuhlgängien', 'behindertengerechten Betten', 'rollstuhl',
       'ROLLSTUHL'], dtype=object)

In [24]:
df_thresh["match_type"].value_counts()

match_type
embedding    381949
keyword          79
Name: count, dtype: int64

In [25]:
df_thresh.to_parquet("/Users/lorenaraichle/Developer/ABSA/PyABSA/results/POC/barriere.parquet")

In [24]:
import pandas as pd
df_thresh = pd.read_parquet("/Users/lorenaraichle/Developer/ABSA/PyABSA/results/POC/barriere.parquet")

### check coverage 

In [25]:
# How many keyword aspects are present in the retrieved results?
retrieved_keyword_aspects = set(df_thresh[df_thresh['match_type'] == 'keyword']['aspect'].str.lower().unique())
coverage_ratio = len(retrieved_keyword_aspects) / len(keywords_barrierefreiheit)
print(f"Coverage ratio: {coverage_ratio:.2%} ({len(retrieved_keyword_aspects)}/{len(keywords_barrierefreiheit)})")


Coverage ratio: 766.67% (23/3)


### check keyword based matches and extend to aspect_keyword as a match type

In [32]:
df_embd = df_thresh[df_thresh['match_type'] == 'embedding']
df_embd.shape

(381949, 9)

In [33]:
df_embd_strict = df_embd[
    (df_embd["match_type"] == "embedding") & 
    (df_embd["distance"] < 0.4)
]
df_embd_strict.shape

(220, 9)

In [29]:
df_embd_cut = df_thresh[(df_thresh['match_type'] == 'embedding') & (df_thresh['distance'] >= 0.4)]
df_embd_cut.shape

(381729, 9)

In [20]:
df_embd_cut["review_id"].nunique() 
df_embd_cut_list = df_embd_cut["review_id"].unique().tolist()
print(len(df_embd_cut_list))

262605


review_sql = f"""
    SELECT id, review
    FROM google_maps_reviews
    WHERE id IN ({review_ids_sql_list})
    AND ({ilike_clauses})
"""

In [39]:

review_ids_sql_list = ", ".join([str(rid) for rid in df_embd_cut_list])
ilike_clauses = " OR ".join([f"review ILIKE '%{kw}%'" for kw in keywords_barrierefreiheit])


review_sql = f"""
    SELECT id, review
    FROM google_maps_reviews
    WHERE {ilike_clauses}
"""


conn = psycopg2.connect(
    host=os.getenv("DB_HOST"),
    port=os.getenv("DB_PORT"),
    dbname=os.getenv("DB_NAME"),
    user=os.getenv("DB_USER"),
    password=os.getenv("DB_PASSWORD"),
)

with conn.cursor() as cur:
    cur.execute(review_sql)
    review_rows = cur.fetchall()

conn.close()

df_review_matches = pd.DataFrame(review_rows, columns=["id", "review"])
df_review_matches.shape


(1560, 2)

In [40]:
df_review_matches

,id,review
0,7381,Es ist ein sehr schönes Bad und vorallem der A...
1,25699,"Obwohl stand nimmt neue Patienten an ,wurde me..."
2,36866,👩‍🦽👩‍🦯 Schönes Lokal mit guter schweizer Küche...
3,42284,Seit über 20 Jahren nehme ich Kosmetik Nicole ...
4,54296,"Super liebes Team, welches alle Wünsche und Be..."
...,...,...
1555,6614378,"Die Höhle ist gut zugänglich, aber nicht Rolls..."
1556,6614896,Gut erreichbar mit öV. Von Bushaltestelle Beat...
1557,6615052,Da ich schlecht zu Fuss bin oder\nFür Rollstuh...
1558,6617395,"Sehr schön, mir gefiel vorallem den Aussenbere..."


In [42]:
# Step 1: Cut df_thresh — embedding < 0.4
df_thresh_strict = df_thresh[
    (df_thresh["match_type"] != "embedding") |
    ((df_thresh["match_type"] == "embedding") & (df_thresh["distance"] < 0.4))
]

# Step 2: Prepare df_new_cut from df_review_matches
df_new_cut = df_review_matches.copy()

df_new_cut["review_id"] = df_new_cut["id"] 
df_new_cut["aspect"] = df_new_cut["review"].apply(lambda x: None)
df_new_cut["aspect_id"] = df_new_cut["review_id"].astype(str) + "_cutkw"
df_new_cut["sentiment"] = None
df_new_cut["confidence"] = None
df_new_cut["snippet_4"] = None
df_new_cut["distance"] = None
df_new_cut["match_type"] = "cutembd_keyword"

# Step 3: Align columns
common_cols = df_thresh.columns.intersection(df_new_cut.columns)
# Step 4: Merge to df_combined
df_combined = pd.concat([df_thresh_strict[common_cols], df_new_cut[common_cols]], ignore_index=True)

# Step 5: Print results
print("Combined match_type counts:")
print(df_combined["match_type"].value_counts())

print(f"\nTotal unique reviews in combined DF: {df_combined['review_id'].nunique()}")


Combined match_type counts:
match_type
cutembd_keyword    1560
embedding           220
keyword              79
Name: count, dtype: int64

Total unique reviews in combined DF: 1634


/var/folders/31/zwpt5b0137974fw30jhhrvtw0000gn/T/ipykernel_21847/2460270078.py:22: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  df_combined = pd.concat([df_thresh_strict[common_cols], df_new_cut[common_cols]], ignore_index=True)


In [43]:
df_combined["review_id"].nunique()


1634

In [44]:
df_combined["sentiment"].value_counts(dropna=False)


sentiment
None        1560
Positive     194
Negative      87
Neutral       18
Name: count, dtype: int64

In [45]:
df_combined.groupby("match_type")["review_id"].nunique()


match_type
cutembd_keyword    1560
embedding           180
keyword              70
Name: review_id, dtype: int64